In [4]:
from langchain_ollama import ChatOllama
from langchain.tools import tool
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
from langchain.agents import create_agent
model = ChatOllama(model='qwen2.5:3b')
@tool
def weather_search(location:str)->str:
    """Get weather forecast for a location."""
    return f"It's cloudy in {location} with temperatures between 23 degree and 11 degree celsius."
model_with_tools = model.bind_tools([weather_search])
messages = [{'role':'user','content':"What's the weather like in Patna?"}]
response = model_with_tools.invoke(messages)
# response = model_with_tools.invoke("What's the weather like in Patna?")
# response
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"args: {tool_call['args']}")


Tool: weather_search
args: {'location': 'Patna'}


In [6]:
messages.append(response)
for tool_call in response.tool_calls:
    if tool_call['name'] == 'weather_search':
        tool_result = weather_search.invoke(tool_call)
    messages.append(tool_result)

print(messages)
final_res = model_with_tools.invoke(messages)
final_res.content

[{'role': 'user', 'content': "What's the weather like in Patna?"}, AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-01-23T08:17:27.8425501Z', 'done': True, 'done_reason': 'stop', 'total_duration': 18770228700, 'load_duration': 5851064200, 'prompt_eval_count': 154, 'prompt_eval_duration': 10476497700, 'eval_count': 21, 'eval_duration': 2387330700, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019be9ed-b6d0-7253-a911-800f0617aad3-0', tool_calls=[{'name': 'weather_search', 'args': {'location': 'Patna'}, 'id': '86f9d0c5-506e-45b0-ba10-0b43139f8899', 'type': 'tool_call'}], usage_metadata={'input_tokens': 154, 'output_tokens': 21, 'total_tokens': 175}), ToolMessage(content="It's cloudy in Patna with temperatures between 23 degree and 11 degree celsius.", name='weather_search', tool_call_id='86f9d0c5-506e-45b0-ba10-0b43139f8899')]


'The weather in Patna is currently cloudy with temperatures ranging from 23 degrees to 11 degrees Celsius.'

In [7]:
# @tool("population_search")
@tool
def get_population(location:str)->str:
    """Get population of a location."""
    return f"..."
mess = [{'role':'user', 'content':'What is the population of Pune?'}]
model_with_tools = model.bind_tools([weather_search, get_population])
result = model_with_tools.invoke(mess)
for tool_call in result.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"args: {tool_call['args']}")

Tool: get_population
args: {'location': 'Pune'}


In [8]:
# from langchain.tools import ToolRuntime
# def summarize_conversation(runtime:ToolRuntime)->str:
#     """Summarize the conversation so far."""
#     messages = runtime.state['messages']
#     human_mess = sum(1 for m in messages if m.__class__.__name__ == 'HumanMessage')
#     ai_mess = sum(1 for m in messages if m.__class__.__name__ == 'AIMessage')
#     tool_mess = sum(1 for m in messages if m.__class__.__name__ == 'ToolMessage')
#     return f"Conversation has {human_mess} user messages, {ai_mess} ai responses adn {tool_mess} tool messages"

# # model_with_tools = model.bind_tools([summarize_conversation])
# res = model.invoke("What's the weather like also summarize the conversation so far.")
# # res = model_with_tools.invoke("What's the other name for India also summarize the conversation so far.")
# res

In [ ]:
from typing import Any
from langgraph.store.memory import InMemoryStore
from langchain.tools import ToolRuntime

@tool
def get_user_info(user_id:str, runtime:ToolRuntime)->str:
    """Get information about users."""
    store = runtime.store
    user_info = store.get(('users',), user_id)
    return str(user_info.value) if user_info else "Unknown User"

@tool
def set_user_info(user_id:str, user_info:dict[str, Any], runtime:ToolRuntime)->str:
    """Save information of the user."""
    store = runtime.store
    store.put(("users",), user_id, user_info)
    return "Saved user succesfully"

agent = create_agent(
    model=model,
    tools=([get_user_info, set_user_info]),
    store=InMemoryStore()
)
agent.invoke({
    'messages':[{'role':'user', 'content':'Save this user information:user_id:me123, name:me, age:20, email:me@vscode.com'}]
    })


{'messages': [HumanMessage(content='Save this user information:user_id:me123, name:me, age:20, email:me@vscode.com', additional_kwargs={}, response_metadata={}, id='df04cba0-4d6d-43d9-8945-e553875c2063'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-01-23T08:22:47.4532692Z', 'done': True, 'done_reason': 'stop', 'total_duration': 18395025900, 'load_duration': 184663900, 'prompt_eval_count': 224, 'prompt_eval_duration': 11315155800, 'eval_count': 53, 'eval_duration': 6797023000, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019be9f2-99bb-7cb1-8f16-1121ed92488a-0', tool_calls=[{'name': 'set_user_info', 'args': {'user_id': 'me123', 'user_info': {'age': 20, 'email': 'me@vscode.com', 'name': 'me'}}, 'id': '5adf6a1b-c6ab-43ae-b2dc-d0b8fc6f82f9', 'type': 'tool_call'}], usage_metadata={'input_tokens': 224, 'output_tokens': 53, 'total_tokens': 277}),
  ToolMessage(content='Saved user succesfully', name='set_use

In [15]:
ress = agent.invoke({
    'messages':[{'role':'user', 'content':'Find user with user_id:me123'}]
})
ress['messages'][-1].content

'The user with the ID me123 has the following information:\n- Name: me\n- Email: me@vscode.com\n- Age: 20'

In [21]:
from dataclasses import dataclass
user_database = {
    "user123":{
        'name':'Chandler Bing',
        'account_type':'premium',
        'balance':5000, 
        'email':'chandler@bing.com'
    },
    'user234':{
        'name':'Ross Geller',
        'account_type':'standard',
        'balance':2000,
        'email':'ross@gelller.com'
    }
}
@dataclass
class UserContext:
    user_id:str

@tool 
def get_account_info(runtime:ToolRuntime[UserContext])->str:
    """Get account information of current user."""
    user_id = runtime.context.user_id
    if user_id in user_database:
        user = user_database[user_id]
        return f"Account Holder: {user['name']}\nAccount Type:{user['account_type']}\nBalance:{user['balance']}\nEmail:{user['email']}"
    return "user_not_found"

agent = create_agent(
    model=model,
    tools=([get_account_info]),
    context_schema=UserContext,
    system_prompt = 'You are a financial assistant.'
)
result=agent.invoke({
    'messages':[{'role':'user', 'content':'Show my account infomation'}]},
    context = UserContext(user_id='user123')
)
result['messages'][-1].content

'Here is your account information:\n\n- Account Holder: Chandler Bing\n- Account Type: premium\n- Balance: $5,000\n- Email: chandler@bing.com'